In [9]:
from collections import Counter
from pathlib import Path
from openpyxl import load_workbook, Workbook
from openpyxl.worksheet.worksheet import Worksheet

from zero_data.io_list import IOResult
from zero_data.io_list.readers.marpower import MarpowerReader
import polars as pl

# Read in IO Lists

In [6]:
io_result_amcs = MarpowerReader().read_io_list([Path("../io_lists/52422003_3210_AMCS IO-List R2.14.xlsx")])
df_amcs = io_result_amcs.io_list
df_amcs

device,tag,yard_tag,target_type,terminal,cabinet,system,description,unit,precision,data_type,mqtt_topic,mqtt_json_path
str,str,str,str,str,str,str,str,str,null,str,str,str
"""KEB1""","""GFDEnabled""",null,"""Bool""","""0""","""DSK_SWR""","""450000 AMCS""","""MTS AMCS KEB1 Ground Fault Dia…",null,null,"""BOOLEAN""","""450000 AMCS/PWR""","""$.GFDEnabled"""
"""KEB1""","""FieldVoltageAvailable""",null,"""Bool""","""2""","""DSK_SWR""","""450000 AMCS""","""MTS AMCS KEB1 Voltage Availabl…",null,null,"""BOOLEAN""","""450000 AMCS/PWR""","""$.FieldVoltageAvailable"""
"""KEB1""","""PA24V""",null,"""Bool""","""4""","""DSK_SWR""","""450000 AMCS""","""MTS AMCS KEB1 Power Distributi…",null,null,"""BOOLEAN""","""450000 AMCS/PWR""","""$.PA24V"""
"""KEB1""","""MA24V""",null,"""Bool""","""5""","""DSK_SWR""","""450000 AMCS""","""MTS AMCS KEB1 Power Distributi…",null,null,"""BOOLEAN""","""450000 AMCS/PWR""","""$.MA24V"""
"""KEB1""","""PA0V""",null,"""Bool""","""6""","""DSK_SWR""","""450000 AMCS""","""MTS AMCS KEB1 Power Distributi…",null,null,"""BOOLEAN""","""450000 AMCS/PWR""","""$.PA0V"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""KEB1""","""CAM2aft_Preset3HmiColor""",null,"""String""",null,"""+CB.1""","""450000 UNDERWLIGHTS""","""CAM2aftPreset3HmiColor""","""#""",null,"""STRING""","""450000 UNDERWLIGHTS/CAM2aft""","""$.Preset3HmiColor"""
"""KEB1""","""CAM2aft_Preset4HmiColor""",null,"""String""",null,"""+CB.1""","""450000 UNDERWLIGHTS""","""CAM2aftPreset4HmiColor""","""#""",null,"""STRING""","""450000 UNDERWLIGHTS/CAM2aft""","""$.Preset4HmiColor"""
"""KEB1""","""CAM2aft_Preset5HmiColor""",null,"""String""",null,"""+CB.1""","""450000 UNDERWLIGHTS""","""CAM2aftPreset5HmiColor""","""#""",null,"""STRING""","""450000 UNDERWLIGHTS/CAM2aft""","""$.Preset5HmiColor"""


## Check tag duplicates

In [7]:
def check_tag_duplicates(io_result: IOResult):
    df = io_result.io_list
    tag_counts = df.group_by("tag").len("tag_count").sort("tag_count", descending=True)
    duplicates = tag_counts.filter(tag_counts["tag_count"] > 1)
    return duplicates

duplicates_amcs = check_tag_duplicates(io_result_amcs)
duplicates_amcs

tag,tag_count
str,u32
"""PT11F01_Power_Factor_Phase_A""",3
"""PT20F01_Active_Power_Phase_B""",3
"""PT41F01_Active_Power_Phase_C""",3
"""PT24F01_RMS_Current_Phase_B""",3
"""PT22F01_RMS_Current_Phase_A""",3
…,…
"""PT34F01_RMS_Current_Phase_C""",2
"""PT53F01_Active_Power_Phase_B""",2
"""PT35F01_RMS_Voltage_C_N""",2


## Schema For nested topics

In [8]:
def get_non_matching_nested_topics(io_result: IOResult, topic_prefix: str):
    power_tag_df =  io_result.io_list.filter(pl.col("mqtt_topic").str.starts_with(topic_prefix))
    nr_of_fields = power_tag_df.group_by(pl.col("mqtt_topic")).len("nr_of_fields").group_by("nr_of_fields").len("len").sort("len", descending=True).head(1).to_dict()["nr_of_fields"][0]

    non_matching_topics = [t for t in io_result.topics if len(t.fields) !=nr_of_fields]
    return non_matching_topics

get_non_matching_nested_topics(io_result_amcs, "power-tag/")

[IOTopic(topic='marpower/300000-dirty-oil/dirty_oil_sludge_tank', fields=[IOValue(name='LEVEL', data_type='REAL')]),
 IOTopic(topic='marpower/450000-firedetection/fire_201h36', fields=[IOValue(name='alarm', data_type='BOOLEAN'), IOValue(name='prealarm', data_type='BOOLEAN'), IOValue(name='fault', data_type='BOOLEAN'), IOValue(name='disabled', data_type='BOOLEAN'), IOValue(name='test', data_type='BOOLEAN')]),
 IOTopic(topic='marpower/450000-navigation-lights/nav_lights', fields=[IOValue(name='COMMONALARM', data_type='BOOLEAN')]),
 IOTopic(topic='marpower/250000-fresh-water/coldwater_water_softner', fields=[IOValue(name='FLOW', data_type='REAL')]),
 IOTopic(topic='marpower/350000-ventilation/gal_vent', fields=[IOValue(name='EMERGSTOPACTIVE', data_type='BOOLEAN')]),
 IOTopic(topic='marpower/450000-firedetection/fire_201h51s', fields=[IOValue(name='alarm', data_type='BOOLEAN'), IOValue(name='prealarm', data_type='BOOLEAN'), IOValue(name='fault', data_type='BOOLEAN'), IOValue(name='disabled

## Internal validation

validate columns against relevant internal list and project definitions:
- System
- DataType

In [51]:
def headers(sheet: Worksheet) -> list[str]:
    return [c[0] for c in sheet.iter_cols(max_row=1) if c]

def iterate_excel_column(sheet: Worksheet, index: int, skip_extra_headers=0) -> list[str]:
    return [c.value for c in next(sheet.iter_cols(min_col= index+skip_extra_headers, max_col=index, min_row=2)) if c and c.value]

def get_definitions_from_workbook_tab(sheet: Worksheet) -> dict[str, list[str]]:
    _headers = headers(sheet)
    return { str(header.value): iterate_excel_column(sheet, header.col_idx) for header in _headers }

def get_definitions(workbook: Workbook) -> dict[str, list[str]]:
    project_definitions = get_definitions_from_workbook_tab(workbook["Project definitions"])
    internal_lists = get_definitions_from_workbook_tab(workbook["Internal lists"])
    return  internal_lists | project_definitions

def validate_column(sheet: Worksheet, column_name: str, expected_values: list[str]) -> dict[str, int]:
    _headers = headers(sheet)
    col_idx = [h for h in _headers if h.value == column_name][0].col_idx
    column_values = iterate_excel_column(sheet, col_idx, skip_extra_headers=0)
    invalid_values = [v for v in column_values if v not in expected_values]
    return dict(Counter(invalid_values))



In [36]:
amcs_path = Path("../io_lists/52422003_3210_AMCS IO-List R2.14.xlsx")
amcs_workbook = load_workbook(amcs_path, data_only=True)
amcs_definitions = get_definitions(amcs_workbook)

pms_path = Path("../io_lists/52422003_3211_PMS IO-List R2.6.xlsx")
pms_workbook = load_workbook(pms_path, data_only=True)
pms_definitions = get_definitions(pms_workbook)

print(pms_definitions)
print(amcs_definitions)

{'IO Type': ['DI', 'DO', 'AI', 'AO', 'SE'], 'Data Type': ['Bool', 'Int16', 'Int32', 'Int64', 'UInt16', 'UInt32', 'UInt64', 'Float', 'Double', 'DateTime', 'AsciiString', 'String', 'Blob'], 'Alert Priority': ['Emergency', 'Alarm', 'Warning', 'Caution', 'Event'], 'Lock Operator': ['<', '>', '='], 'True False': ['True', 'False', 'FAT'], 'Direction': ['In', 'InOut', 'Out'], 'Module Type': ['Fieldbus', '750-403', '750-430', '750-450#02', '750-450#04', '750-454', '750-455', '750-461', '750-464#02', '750-464#04', '750-469/003-000', '750-477', '750-485', '750-496', '750-517', '750-530', '750-554', '750-555', '750-559', '750-610', '750-652', '750-655', '750-658#8', '750-658#12', '750-658#16', '750-658#20', '750-658#24', '750-658#32', '750-658#40', '750-658#48', '750-1400', '750-1415', '750-1425', '750-1500', '750-1515', '750-511/000-001'], 'Modbus Data Type': ['Coil', 'Input', 'InputRegister', 'HoldingRegister'], 'Devices': ['KEB4', 'KEB5', 'KEB6', 'KEB7', 'KEB8', 'KEB9'], 'Cabinet': ['+CB.1', '

In [62]:
amcs_io_sheet = amcs_workbook["IO-List"]
print("AMCS System Column")
print(validate_column(amcs_io_sheet, "System", amcs_definitions["Systems"]))
print("AMCS Data Type Column")
print(validate_column(amcs_io_sheet, "Target Type", amcs_definitions["Data Type"]))

AMCS System Column
{'450000 AMCS': 78, 'SPARE': 86, '210000 BILGE FIFI': 48, '090000 DOORS HATCHES': 58, '250000 FRESH WATER': 61, '340000 SEWAGE': 20, '750000 DECK EQUIP': 6, 'THERMODYNAMICA': 1, '220000 NOVEC': 4, '350000 VENTILATION': 53, '250000 TECHWATER': 23, 'HPU': 4, 'KVM SWITCHING': 18, '450000 24VDC SYSTEM': 22, '380000 SEAWATER': 9, '150000 PCS': 24, '450000 BURGLAR': 7, '290000 PNEUMATIC ': 1, '300000 DIRTY OIL': 1, '150000 PROPULSION': 4, '210000 GENERAL SERVICE': 3, '450000 NAVIGATION LIGHTS': 81, '170000 STEERING SYSTEM': 5, '450000 GAS DETECTION': 2, '450000 FIREDETECTION': 67, 'AC DISTR 10P0.1': 885, 'AC DISTR 10P0.3': 705, 'AC DISTR 10P1': 480, 'AC DISTR 10P2': 375, 'AC DISTR 10P3': 945, '450000 UNDERWLIGHTS': 288}
AMCS Data Type Column
{'Uint32': 28, 'Unit16': 61}


{'Uint32': 28, 'Unit16': 61}

In [61]:
pms_io_sheet = pms_workbook["IO-List"]
print("PMS System Column")
print(validate_column(pms_io_sheet, "System", amcs_definitions["Systems"]))
print("PMS Data Type Column")
print(validate_column(pms_io_sheet, "Target", amcs_definitions["Data Type"]))


PMS System Column
{'450000 DC DISTRIBUTION': 206, '150000 PROPULSION': 68, 'SPARE': 81, '450000 AMCS': 10, '450000 MAIN POWER STORAGE': 9856, '450000 DYNAMIC CONV PS': 72, '450000 DYNAMIC CONV': 2, '450000 UGRID ': 9, '210000 BILGE FIFI': 90, '380000 SEAWATER COOLING': 90, '450000 HVAC': 36, '450000 DYNAMIC CONV SB': 72, '270000 DOMESTIC EQUIPM': 16}
PMS Data Type Column
{'Type': 1}


In [59]:
[h.value for h in headers(pms_io_sheet)]

['Rev.',
 'Deleted',
 'Device',
 'Module',
 None,
 'Terminal',
 'Is Subscribe',
 'Prefix Device',
 'Tag',
 'OPC UA',
 None,
 'System',
 'Description',
 'Parent',
 'Redundant tag',
 'Target',
 None,
 'Cabinet',
 'Yard Tag',
 'P&ID',
 'Sensor',
 'MakerSupplier',
 'Cable',
 None,
 None,
 'Mqtt',
 None,
 None,
 'Modbus',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 'Range',
 None,
 'Unit',
 'Precision',
 'Alert Code',
 'Alert',
 None,
 None,
 None,
 None,
 'Delay On',
 'Acknowledge Location',
 'Sounding Locations',
 'Alert priority',
 'Intended Operator Response',
 'Group Alarm',
 'Vdr ID',
 'Call GEA on Alert ',
 'Disallow Inhibit',
 'Category A',
 'General Lock',
 None,
 None,
 'HH Lock',
 None,
 None,
 'H Lock',
 None,
 None,
 'L Lock',
 None,
 None,
 'LL Lock',
 None,
 None,
 'Do Not Log',
 'Log To Daily Report',
 'Log to CDP',
 'Workstation',
 'Timestamp',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,